# Modelo 2 — AGCRN bimodal (SECA × PICO) · Season 1 · dados corrigidos

Versão enxuta do `Dengue_final_ufvjm.ipynb` só com o necessário para treinar e avaliar o **Modelo 2** sobre os bancos
`train1_FINAL_NN.parquet` / `test1_FINAL_NN.parquet` gerados por `refazer_season1.py` (pasta `S1_corrigido/`).

O código do modelo, das losses, do treino, da submissão, do MinT dinâmico, da avaliação e do gráfico é **o mesmo do notebook original** (células 22 e 41–50).
Mudanças marcadas com `# [NOVO]`:

| Mudança | Por quê |
|---|---|
| Caminhos em uma célula de configuração (Drive / Kaggle / local) | Tudo o que é gerado (`.db`, pesos, gráfico, checkpoints) vai para o Drive e sobrevive se a sessão cair |
| Semente fixa (`SEED`) | Permite comparar execuções (dados antigos × novos) |
| Resultados da season são **substituídos** no SQLite, não acumulados | Antes, rodar 2× misturava as execuções nas métricas |
| Checkpoint a cada época + retomada automática (`RETOMAR = True`) | Se a sessão cair, basta reconectar e rodar todas as células de novo: o treino continua da última época salva |
| AGCRN embutida, sem `pip install` | O Colab atual não tem mais torch 2.4; instalar torch-geometric-temporal obrigava a compilar torch-scatter/sparse |
| Pesos do especialista SECA salvos mesmo sem validação | Antes, a SECA nunca tinha validação (SE 16–25 são todas de pico) e seus pesos não eram salvos |

**Como usar:** coloque os dois parquets numa pasta do Drive (padrão: `MyDrive/dengue_S1/`), escolha *Ambiente de execução → Alterar tipo → GPU T4* e rode as células em ordem (ou *Executar tudo*). **Não há nada para instalar**: a AGCRN está embutida (mesma implementação do `torch_geometric_temporal`, compatível com os pesos `.pt` já salvos) e usa o PyTorch do próprio Colab.
Tempo esperado na T4 gratuita: **~1h45 a 2h45** (a SECA roda as 200 épocas inteiras, ~21 s/época).

## 1. Pacotes

In [ ]:
# @title Conferir ambiente (não é preciso instalar nada)
# A versão anterior instalava torch 2.4 + torch-geometric-temporal. O Colab atual não tem mais o torch 2.4
# e tentava COMPILAR torch-scatter/torch-sparse (dezenas de minutos). Agora a AGCRN está embutida
# no notebook (célula 1b) e usa o PyTorch que já vem no Colab.
import torch, numpy, pandas, scipy, pyarrow, sys
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| numpy', numpy.__version__,
      '| pandas', pandas.__version__, '| pyarrow', pyarrow.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA — ative a GPU T4')

In [ ]:
# @title Importar pacotes
import os, sys, gc, time, random, sqlite3, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import nbinom, pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore', category=FutureWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Hardware: {device.type.upper()}", torch.cuda.get_device_name(0) if device.type == 'cuda' else '')
if device.type != 'cuda':
    print("⚠️ SEM GPU: o treino levaria dias. Ative a GPU em 'Ambiente de execução → Alterar tipo'.")

In [ ]:
# @title AGCRN (mesma implementação de torch_geometric_temporal.nn.recurrent.AGCRN, só PyTorch)
# Embutida aqui para não depender de torch-geometric-temporal / torch-scatter / torch-sparse,
# que exigem versões antigas do torch e precisam ser compiladas no Colab atual.
import math


def _glorot(t):
    a = math.sqrt(6.0 / (t.size(-2) + t.size(-1)))
    with torch.no_grad():
        t.uniform_(-a, a)


class AVWGCN(nn.Module):
    """Adaptive Graph Convolution com pesos por nó (Bai et al., 2020)."""
    def __init__(self, in_channels: int, out_channels: int, K: int, embedding_dimensions: int):
        super().__init__()
        self.K = K
        self.weights_pool = nn.Parameter(torch.empty(embedding_dimensions, K, in_channels, out_channels))
        self.bias_pool = nn.Parameter(torch.empty(embedding_dimensions, out_channels))
        _glorot(self.weights_pool)
        nn.init.zeros_(self.bias_pool)

    def forward(self, X: torch.Tensor, E: torch.Tensor) -> torch.Tensor:
        number_of_nodes = E.shape[0]
        supports = F.softmax(F.relu(torch.mm(E, E.transpose(0, 1))), dim=1)
        support_set = [torch.eye(number_of_nodes, device=supports.device, dtype=supports.dtype), supports]
        for _ in range(2, self.K):
            support_set.append(torch.matmul(2 * supports, support_set[-1]) - support_set[-2])
        supports = torch.stack(support_set, dim=0)
        W = torch.einsum('nd,dkio->nkio', E, self.weights_pool)
        bias = torch.matmul(E, self.bias_pool)
        X_G = torch.einsum('knm,bmc->bknc', supports, X)
        X_G = X_G.permute(0, 2, 1, 3)
        return torch.einsum('bnki,nkio->bno', X_G, W) + bias


class AGCRN(nn.Module):
    """Célula recorrente AGCRN (GRU com convolução de grafo adaptativa)."""
    def __init__(self, number_of_nodes: int, in_channels: int, out_channels: int, K: int, embedding_dimensions: int):
        super().__init__()
        assert K > 1, 'K deve ser > 1'
        self.number_of_nodes = number_of_nodes
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.K = K
        self.embedding_dimensions = embedding_dimensions
        self._gate = AVWGCN(in_channels + out_channels, 2 * out_channels, K, embedding_dimensions)
        self._update = AVWGCN(in_channels + out_channels, out_channels, K, embedding_dimensions)

    def forward(self, X: torch.Tensor, E: torch.Tensor, H: torch.Tensor = None) -> torch.Tensor:
        if H is None:
            H = torch.zeros(X.shape[0], X.shape[1], self.out_channels, device=X.device, dtype=X.dtype)
        X_H = torch.cat((X, H), dim=-1)
        Z_R = torch.sigmoid(self._gate(X_H, E))
        Z, R = torch.split(Z_R, self.out_channels, dim=-1)
        C = torch.cat((X, Z * H), dim=-1)
        HC = torch.tanh(self._update(C, E))
        return R * H + (1 - R) * HC

## 2. Configuração, caminhos e variáveis

In [ ]:
# @title Configuração  # [NOVO]
SEASON_ALVO = 1
SEED = 42                      # fixa a inicialização da rede (use o mesmo valor para comparar execuções)
RETOMAR = True                 # True: se existir checkpoint desta season, continua dele. False: treina do zero
CKPT_A_CADA_EPOCAS = 1         # salva checkpoint local a cada N épocas (cópia no Drive a cada 5)
EPOCHS_MAX = 200               # mesmos valores da execução original (célula 48)
PATIENCE = 25
USAR_TORCH_COMPILE = True      # se der erro mencionando 'inductor'/'dynamo'/'compile', mude para False e rode de novo

if not USAR_TORCH_COMPILE:
    torch.compile = lambda modelo, *a, **k: modelo   # roda sem compilação (um pouco mais lento, mesmo resultado)

try:
    import google.colab  # noqa
    EM_COLAB = True
except ImportError:
    EM_COLAB = False
EM_KAGGLE = os.path.exists('/kaggle/working')

if EM_KAGGLE:
    PASTA_DADOS = '/kaggle/input/dengue-s1'          # nome do Dataset que você criar no Kaggle
    PASTA_SAIDA = '/kaggle/working'
elif EM_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    PASTA_DADOS = '/content/drive/MyDrive/dengue_S1'                     # onde estão os 2 parquets
    PASTA_SAIDA = '/content/drive/MyDrive/dengue_S1/resultados_modelo2'  # tudo o que for gerado
else:
    PASTA_DADOS = '.'
    PASTA_SAIDA = './resultados_modelo2'

# [NOVO] Trabalhar no DISCO LOCAL da sessão. O Drive só é usado para (1) ler os dados uma vez e
# (2) guardar cópias de checkpoints/resultados. Se o Drive cair, o treino continua.
import shutil
PASTA_DADOS, PASTA_SAIDA = os.path.abspath(PASTA_DADOS), os.path.abspath(PASTA_SAIDA)
PASTA_LOCAL = '/content/trabalho' if EM_COLAB else PASTA_SAIDA
os.makedirs(PASTA_LOCAL, exist_ok=True)
os.makedirs(PASTA_SAIDA, exist_ok=True)
for f in (f"train{SEASON_ALVO}_FINAL_NN.parquet", f"test{SEASON_ALVO}_FINAL_NN.parquet"):
    origem, destino = os.path.join(PASTA_DADOS, f), os.path.join(PASTA_LOCAL, f)
    assert os.path.exists(origem) or os.path.exists(destino), f"Não encontrei {f} em {PASTA_DADOS}"
    if not os.path.exists(destino):
        print(f"📥 Copiando {f} do Drive para o disco local ...")
        shutil.copy(origem, destino)
# traz checkpoints/banco já existentes no Drive (para RETOMAR depois de uma queda)
for f in os.listdir(PASTA_SAIDA):
    if (f.startswith('ckpt_') or f == 'resultados_agcrn.db') and not os.path.exists(os.path.join(PASTA_LOCAL, f)):
        shutil.copy(os.path.join(PASTA_SAIDA, f), os.path.join(PASTA_LOCAL, f))
PASTA_DADOS = PASTA_LOCAL
os.chdir(PASTA_LOCAL)
DB_LOCAL = os.path.join(PASTA_LOCAL, 'resultados_agcrn.db')


def salvar_no_drive(arquivo):
    """Copia um arquivo local para o Drive. Se o Drive estiver fora do ar, avisa e segue."""
    if PASTA_LOCAL == PASTA_SAIDA:
        return
    try:
        shutil.copy(arquivo, os.path.join(PASTA_SAIDA, os.path.basename(arquivo)))
    except OSError as e:
        print(f"   ⚠️ não consegui copiar {arquivo} para o Drive ({e}); o treino continua")


print('📂 Trabalhando em:', os.getcwd())
print('💾 Cópias no Drive:', PASTA_SAIDA)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
# @title Variáveis do banco de dados (célula 3 do notebook original)
VARS = {
    'municipio':    'geocode',
    'semana_epi':   'epiweek',
    'data':         'date',
    'casos_dengue': 'casos.x',
    'populacao':    'population',
    'uf_code':      'uf_code',
}
SEASONS = {
    1: {'train': "train1_dengue.parquet", 'test': "test1_dengue.parquet", 'label': '2022-2023'},
    2: {'train': "train2_dengue.parquet", 'test': "test2_dengue.parquet", 'label': '2023-2024'},
    3: {'train': "train3_dengue.parquet", 'test': "test3_dengue.parquet", 'label': '2024-2025'},
    4: {'train': "train4_dengue.parquet", 'test': "test4_dengue.parquet", 'label': '2025-2026'},
}
QUANTIS = {'50': (0.25, 0.75), '80': (0.10, 0.90), '90': (0.05, 0.95), '95': (0.025, 0.975)}

## 3. Conciliação município → estado (MinT dinâmico, célula 22 do original)

In [ ]:
# @title Reconciliar Mint Dinâmico
import numpy as np
import pandas as pd

def reconciliar_mint_dinamico(df_municipal: pd.DataFrame, df_estadual: pd.DataFrame,
                               semana_inicio_alta: int = 47, semana_fim_alta: int = 25,
                               col_uf_code: str = 'uf_code') -> tuple:
    """
    Executa a Reconciliação Hierárquica Mínimo Traço (MinT) Dinâmica e Sazonal
    baseada nas Semanas Epidemiológicas reais de transmissão da Dengue.
    Suporta janelas temporais que cruzam a virada do ano civil (ex: SE-49 a SE-22).
    """
    print(f"⚙️ [Dynamic MinT] Iniciando reconciliação orientada a dados (Alta Transmissão: SE-{semana_inicio_alta} a SE-{semana_fim_alta})...")

    # Lista de colunas probabilísticas que passarão pelo ajuste geométrico
    cols_quantis = ['pred', 'lower_50', 'upper_50', 'lower_80', 'upper_80', 'lower_90', 'upper_90', 'lower_95', 'upper_95']
    datas = sorted(df_municipal['date'].unique())

    df_mun_rec = df_municipal.copy()
    df_est_rec = df_estadual.copy()

    # Mapeamentos estruturais da malha geográfica
    mapa_uf = df_municipal[[VARS['municipio'], col_uf_code]].drop_duplicates().set_index(VARS['municipio'])[col_uf_code].to_dict()
    geocodes, ufs = sorted(df_municipal[VARS['municipio']].unique()), sorted(df_estadual[col_uf_code].unique())

    n_uf, n_mun = len(ufs), len(geocodes)
    uf_idx, mun_idx = {uf: i for i, uf in enumerate(ufs)}, {g: i for i, g in enumerate(geocodes)}

    # ==============================================================================
    # 1. CONSTRUÇÃO DA MATRIZ DE ESTRUTURA ESPACIAL (S)
    # ==============================================================================
    S = np.zeros((n_uf + n_mun, n_mun), dtype=np.float32)
    for g in geocodes:
        if mapa_uf.get(g) in uf_idx:
            S[uf_idx[mapa_uf.get(g)], mun_idx[g]] = 1.0
        S[n_uf + mun_idx[g], mun_idx[g]] = 1.0

    # ==============================================================================
    # 2. SEPARAÇÃO SAZONAL POR SEMANA EPIDEMIOLÓGICA (DADOS DE TREINO)
    # ==============================================================================
    df_municipal['datetime_aux'] = pd.to_datetime(df_municipal['date'])
    df_municipal['semana_num_aux'] = df_municipal['datetime_aux'].dt.isocalendar().week

    # 🔥 CORREÇÃO: Tratamento de janelas que cruzam a virada do ano civil
    if semana_inicio_alta <= semana_fim_alta:
        mask_alta = df_municipal['semana_num_aux'].between(semana_inicio_alta, semana_fim_alta)
    else:
        mask_alta = (df_municipal['semana_num_aux'] >= semana_inicio_alta) | (df_municipal['semana_num_aux'] <= semana_fim_alta)

    df_alta = df_municipal[mask_alta]
    df_baixa = df_municipal[~mask_alta]

    # ==============================================================================
    # 3. ALGORITMO INTERNO DE CÁLCULO DA MATRIZ DE PROJEÇÃO MINT (P)
    # ==============================================================================
    def calcular_matriz_P(df_fase):
        variancias = []
        for g in geocodes:
            df_mun_g = df_fase[df_fase[VARS['municipio']] == g]
            if len(df_mun_g) > 1 and 'casos_reais' in df_mun_g.columns:
                erros = df_mun_g['casos_reais'] - df_mun_g['pred']
                # Shrinkage estatístico para evitar divisão por zero em locais esparsos
                variancias.append(max(np.var(erros), 1e-3))
            else:
                variancias.append(1.0)

        var_estados = np.zeros(n_uf)
        for idx_g, g in enumerate(geocodes):
            uf = mapa_uf.get(g)
            if uf in uf_idx:
                var_estados[uf_idx[uf]] += variancias[idx_g]

        # Montagem da matriz diagonal de covariância dos erros (W)
        diag_w = np.concatenate([var_estados, np.array(variancias)])
        W_inv = np.diag(1.0 / diag_w)

        # Resolução da equação de Mínimos Quadrados Ponderados (WLS Estrutural)
        STS_wls = S.T @ W_inv @ S + 1e-6 * np.eye(n_mun)
        return np.linalg.solve(STS_wls, S.T @ W_inv)

    # Pré-computação das duas lentes geométricas do ecossistema
    print("    -> Calculando pesos para o bloco de ALTA TRANSMISSÃO (Picos)...")
    P_alta = calcular_matriz_P(df_alta)

    print("    -> Calculando pesos para o bloco de BAIXA TRANSMISSÃO (Entressafra)...")
    P_baixa = calcular_matriz_P(df_baixa)

    # ==============================================================================
    # 4. LOOP DE RECONCILIAÇÃO TEMPORAL PASSO A PASSO
    # ==============================================================================
    df_mun_rec = df_mun_rec.set_index([VARS['municipio'], 'date'])
    df_est_rec = df_est_rec.set_index([col_uf_code, 'date'])

    for col in cols_quantis:
        if col not in df_municipal.columns:
            continue

        # Pivoteamento para processamento vetorial rápido em matrizes
        mun_pivot = df_municipal.groupby([VARS['municipio'], 'date'])[col].mean().unstack(level='date')
        est_pivot = df_estadual.groupby([col_uf_code, 'date'])[col].mean().unstack(level='date')

        est_matrix = est_pivot.reindex(index=ufs, columns=datas).fillna(0.0).values
        mun_matrix = mun_pivot.reindex(index=geocodes, columns=datas).fillna(0.0).values
        Y_hat_total = np.vstack([est_matrix, mun_matrix])

        # Arrays vazios para armazenar as projeções limpas da coluna atual
        Y_rec_final = np.zeros_like(Y_hat_total[n_uf:, :])
        Y_rec_est_final = np.zeros_like(Y_hat_total[:n_uf, :])

        # Roteamento Sazonal Dinâmico coluna por coluna (Semana por Semana)
        for t_idx, data_str in enumerate(datas):
            semana_corrente = pd.to_datetime(data_str).isocalendar().week
            Y_hat_t = Y_hat_total[:, t_idx:t_idx+1]

            # 🔥 CORREÇÃO: Avaliação robusta do roteamento para janelas invertidas
            if semana_inicio_alta <= semana_fim_alta:
                eh_alta_temporada = (semana_inicio_alta <= semana_corrente <= semana_fim_alta)
            else:
                eh_alta_temporada = (semana_corrente >= semana_inicio_alta) or (semana_corrente <= semana_fim_alta)

            # Seleção do operador de projeção ótimo para a assinatura epidemiológica da semana
            if eh_alta_temporada:
                P_t = P_alta
            else:
                P_t = P_baixa

            # 1. Projeção coerente dos nós base municipais (Garante não-negatividade)
            mun_rec_t = np.maximum(P_t @ Y_hat_t, 0)
            Y_rec_final[:, t_idx:t_idx+1] = mun_rec_t

            # 2. Agregação coerente estrita e imediata para o nível Estadual
            Y_rec_est_final[:, t_idx:t_idx+1] = S[:n_uf, :] @ mun_rec_t

        # Mapeamento reverso das matrizes para o formato longo do Pandas Dataframe
        df_mun_mapped = pd.DataFrame(Y_rec_final, index=geocodes, columns=datas).stack().to_frame(name=col).round(2)
        df_mun_mapped.index.names = [VARS['municipio'], 'date']

        df_est_mapped = pd.DataFrame(Y_rec_est_final, index=ufs, columns=datas).stack().to_frame(name=col).round(2)
        df_est_mapped.index.names = [col_uf_code, 'date']

        # Atualização atômica dos arquivos de saída
        df_mun_rec.update(df_mun_mapped)
        df_est_rec.update(df_est_mapped)

    # Limpeza e remoção das colunas auxiliares de tempo do dataframe de entrada
    df_municipal.drop(columns=['datetime_aux', 'semana_num_aux'], inplace=True, errors='ignore')
    print("✅ [Dynamic MinT] Processo concluído. Hierarquia perfeitamente coerente.")

    return df_mun_rec.reset_index(), df_est_rec.reset_index()

## 4. Modelo 2: funções de perda, features e arquitetura (células 41–45)

In [ ]:
# @title NB loss pico
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import nbinom

def nb_loss_pico(log_mu: torch.Tensor, log_r: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    log_mu = torch.clamp(log_mu, min=-10, max=10)
    log_r  = torch.clamp(log_r,  min=-6,  max=6)
    mu, r = torch.exp(log_mu), torch.exp(log_r)

    # 1. NLL Base (Justa)
    nll = (
        - torch.lgamma(y + r) + torch.lgamma(r) + torch.lgamma(y + 1)
        - r * torch.log(r / (r + mu + 1e-8)) - y * torch.log(mu / (r + mu + 1e-8))
    )

    # 2. Ponderação de Magnitude
    # Faz o gradiente prestar muito mais atenção nas semanas com volume alto
    peso_surto = 1.0 + torch.log1p(y)

    # 3. Penalidade Assimétrica de Subestimação (O "Motor de Aceleração")
    # F.relu(y - mu) só ativa se a realidade (y) for MAIOR que a previsão (mu).
    # Se o modelo tentar prever para baixo durante o surto, ele toma uma "multa" linear pesada.
    erro_subestimacao = F.relu(y - mu)
    penalidade = erro_subestimacao * 0.2  # Fator de agressividade (0.2 a 0.5 costuma ser ideal)

    loss_final = (nll * peso_surto) + penalidade
    return loss_final.mean()

In [ ]:
# @title NB loss seca
def nb_loss_seca(log_mu: torch.Tensor, log_r: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    log_mu = torch.clamp(log_mu, min=-10, max=10)
    log_r  = torch.clamp(log_r,  min=-6,  max=6)
    mu, r = torch.exp(log_mu), torch.exp(log_r)

    # 1. NLL Base (Justa)
    nll = (
        - torch.lgamma(y + r) + torch.lgamma(r) + torch.lgamma(y + 1)
        - r * torch.log(r / (r + mu + 1e-8)) - y * torch.log(mu / (r + mu + 1e-8))
    )

    # 2. Penalidade Assimétrica de Superestimação (O "Escudo Anti-Ruído")
    # F.relu(mu - y) só ativa se o modelo prever MAIS (mu) do que a realidade (y).
    erro_superestimacao = F.relu(mu - y)

    # 3. Trava de Falso Alarme (Zero-Inflation Shield)
    # torch.exp(-y) vale 1.0 quando os casos reais são 0, e cai rapidamente para perto de zero se houver casos reais.
    # Ou seja: A multa só é aplicada cruelmente se a cidade estava sem dengue.
    peso_falso_alarme = torch.exp(-y)

    penalidade = erro_superestimacao * peso_falso_alarme * 0.5 # Fator de conservadorismo

    loss_final = nll + penalidade
    return loss_final.mean()

In [ ]:
# @title Criar Quantis
def nb_quantile(mu: np.ndarray, r: np.ndarray, q: float) -> np.ndarray:
    p = r / (r + mu + 1e-8)
    return nbinom.ppf(q, n=r, p=p).astype(float)

In [ ]:
# @title LISTAS DE FEATURES PARA O AGCRN (Atualizadas com nomes pós-transformação)


# Variáveis físicas básicas que o modelo precisa saber a posição exata
COLS_FISICA = ['velocidade', 'aceleracao']

# 🌵 FEATURES DA SECA (Foco em clima basal e imunidade de longo prazo)
COLS_SECA = COLS_FISICA + [
    'access_count', 'seno_semana', 'temp_min', 'w2_velocidade_t1',
    'w1_log_pop_t1', 'w1_imunidade_t1', 'w2_log_inc_dengue_t1', 'w1_log_inc_dengue_t1',
    'imunidade_ewm', 'imunidade_ewm_t1', 'log_inc_dengue_t1', 'log_inc_chiku',
    'log_pop_t1', 'rel_humid_max', 'log_inc_chiku_t1', 'cosseno_semana',
    'w1_velocidade_t1', 'temp_med', 'koppen_encoded'
]

# 🌊 FEATURES DO PICO (Foco em explosão espacial W1/W2 e aceleração meteorológica)
COLS_PICO = COLS_FISICA + [
    'access_count', 'seno_semana', 'w2_velocidade_t1', 'w1_log_pop_t1',
    'w2_aceleracao_t1', 'access_count_t1', 'w1_log_inc_dengue_t1',
    'imunidade_ewm', 'imunidade_ewm_t1', 'log_inc_dengue_t1', 'log_inc_chiku',
    'log_pop_t1', 'cosseno_semana', 'w1_velocidade_t1', 'w1_aceleracao_t1',
    'w2_log_inc_dengue_t1', 'w2_temp_forecast_t1', 'w1_temp_forecast_t1',
    'log_inc_chiku_t1', 'umid_med_forecast', 'temp_min', 'koppen_encoded',
    'w2_umid_forecast_t1', 'temp_med_forecast', 'w2_precip_forecast_t1'
]

In [ ]:
# @title NOVO AGCRN padrão
import torch
import torch.nn as nn
import torch.nn.functional as F

class PreditorNBAGCRN(nn.Module):
    def __init__(self, num_nodes: int, num_features: int, embedding_dim: int = 64, idx_velocidade: int = 0, idx_aceleracao: int = 1):
        super().__init__()

        # Recebe dinamicamente os índices da velocidade e aceleração
        self.idx_vel = idx_velocidade
        self.idx_acel = idx_aceleracao

        # Node Embeddings (Aprendem o viés estrutural de cada cidade)
        self.node_embeddings = nn.Parameter(torch.randn(num_nodes, embedding_dim), requires_grad=True)

        self.recurrent = AGCRN(
            number_of_nodes=num_nodes,
            in_channels=num_features,
            out_channels=64,
            K=2,                                   # Ordem de vizinhança espacial
            embedding_dimensions=embedding_dim     # Passando a dimensão do embedding
        )

        # 🔥 ALTERAÇÃO: Inclusão de Dropout Espaço-Temporal
        self.dropout = nn.Dropout(p=0.2)

        self.head_mu = nn.Linear(64, 1)
        # Head do parâmetro R recebe o hidden state + as 2 variáveis físicas brutas (vel e acel)
        self.head_logr = nn.Linear(64 + 2, 1)

    def forward(self, x: torch.Tensor, log_pop: torch.Tensor, h=None):
        # 1. Adiciona a dimensão de Batch
        x_batched = x.unsqueeze(0)

        # 2. Passa pela rede recorrente (Geração do estado oculto espaço-temporal)
        h = self.recurrent(x_batched, self.node_embeddings, h)

        h_out = h.squeeze(0)
        h_relu = F.relu(h_out)

        # 🔥 CORREÇÃO: O dropout agora é devidamente aplicado antes de calcular mu e r
        h_reg = self.dropout(h_relu)

        # Cálculo do log_mu utilizando o tensor com Dropout (h_reg)
        log_mu_raw = self.head_mu(h_reg)
        log_mu = log_mu_raw + log_pop.unsqueeze(-1)

        # Resgate da física diretamente da matriz de entrada usando os índices dinâmicos
        vel = x[:, self.idx_vel].unsqueeze(-1)
        acel = x[:, self.idx_acel].unsqueeze(-1)

        # Concatena o estado oculto regularizado (h_reg) com a física
        h_r = torch.cat([h_reg, vel, acel], dim=-1)
        log_r = self.head_logr(h_r)

        return log_mu, log_r, h

## 5. Treino e validação bifurcados (célula 46) e submissão municipal (célula 47)

In [ ]:
# @title NOVO TREINO E VALIDAÇÃO
# ==============================================================================
# FUNÇÃO PRINCIPAL: TREINO E VALIDAÇÃO BIFURCADOS
# ==============================================================================
def treinar_e_validar_season(df_train: pd.DataFrame, df_test: pd.DataFrame,
                             season_id: int, encoder_means_globais: dict,
                             num_nodes: int, mapa_simples: pd.DataFrame,
                             epochs_max: int = 150, patience: int = 25) -> dict:

    torch.backends.cudnn.benchmark = True

    # 🔥 OTIMIZAÇÃO DE RAM 2: Fim do .copy(). Usaremos os ponteiros de memória originais!
    df_tr, df_tgt = df_train, df_test

    if 'log_pop' not in df_tr.columns:
        df_tr['log_pop'] = np.log(df_tr[VARS['populacao']].clip(lower=1).astype(float))
    if 'log_pop' not in df_tgt.columns:
        df_tgt['log_pop'] = np.log(df_tgt[VARS['populacao']].clip(lower=1).astype(float))

    semanas_treino = sorted(df_tr[VARS['semana_epi']].unique())
    df_tr_real = df_tr[df_tr[VARS['semana_epi']].isin(semanas_treino[:-10])]
    df_val_int = df_tr[df_tr[VARS['semana_epi']].isin(semanas_treino[-10:])]

    # --------------------------------------------------------------------------
    # 1. PURGA ABSOLUTA DOS ÍNDICES BUGADOS
    # Removemos node_id antigo e time_index. Ordenamos pela 'data' (inquebrável)
    # --------------------------------------------------------------------------
    df_tr_sort  = df_tr_real.drop(columns=['node_id', 'time_index'], errors='ignore').merge(mapa_simples, on=VARS['municipio'], how='left').sort_values([VARS['data'], 'node_id'])
    df_val_sort = df_val_int.drop(columns=['node_id', 'time_index'], errors='ignore').merge(mapa_simples, on=VARS['municipio'], how='left').sort_values([VARS['data'], 'node_id'])
    df_tgt_sort = df_tgt.drop(columns=['node_id', 'time_index'], errors='ignore').merge(mapa_simples, on=VARS['municipio'], how='left').sort_values([VARS['data'], 'node_id'])

    # --------------------------------------------------------------------------
    # 2. CALENDÁRIO BASEADO NA DATA (RESOLVE O VALUE_ERROR DO RESHAPE)
    # --------------------------------------------------------------------------
    tempos_tr  = sorted(df_tr_sort[VARS['data']].unique())
    tempos_val = sorted(df_val_sort[VARS['data']].unique())
    tempos_tgt = sorted(df_tgt_sort[VARS['data']].unique())

    # Mapeia a semana_epi real correspondente a cada data única para o "Switching" da Loss
    epiweeks_tr  = df_tr_sort.groupby(VARS['data'])[VARS['semana_epi']].first().values % 100
    epiweeks_val = df_val_sort.groupby(VARS['data'])[VARS['semana_epi']].first().values % 100
    epiweeks_tgt = df_tgt_sort.groupby(VARS['data'])[VARS['semana_epi']].first().values % 100

    # Identificação das features de cada regime
    cols_seca = [c for c in COLS_SECA if c in df_tr_sort.columns]
    cols_pico = [c for c in COLS_PICO if c in df_tr_sort.columns]

    num_features_seca = len(cols_seca)
    num_features_pico = len(cols_pico)

    idx_vel_seca  = cols_seca.index('velocidade') if 'velocidade' in cols_seca else 0
    idx_acel_seca = cols_seca.index('aceleracao') if 'aceleracao' in cols_seca else 1
    idx_vel_pico  = cols_pico.index('velocidade') if 'velocidade' in cols_pico else 0
    idx_acel_pico = cols_pico.index('aceleracao') if 'aceleracao' in cols_pico else 1

    # Escalonamento Independente por Regime
    X_tr_sc_seca  = df_tr_sort[cols_seca].fillna(0).values.astype(np.float16)
    X_val_sc_seca = df_val_sort[cols_seca].fillna(0).values.astype(np.float16)
    X_tgt_sc_seca = df_tgt_sort[cols_seca].fillna(0).values.astype(np.float16)

    X_tr_sc_pico  = df_tr_sort[cols_pico].fillna(0).values.astype(np.float16)
    X_val_sc_pico = df_val_sort[cols_pico].fillna(0).values.astype(np.float16)
    X_tgt_sc_pico = df_tgt_sort[cols_pico].fillna(0).values.astype(np.float16)

    # Conversores para GPU
    def _to_tensor(X_sc, tempos, num_f): return torch.tensor(X_sc.reshape(len(tempos), num_nodes, num_f), dtype=torch.float16).to(device)
    def _pop_tensor(df_sort, tempos): return torch.tensor(df_sort['log_pop'].values.reshape(len(tempos), num_nodes), dtype=torch.float16).to(device)
    def _casos_tensor(df_sort, tempos): return torch.tensor(df_sort[VARS['casos_dengue']].fillna(0).values.reshape(len(tempos), num_nodes), dtype=torch.float32).to(device)

    # Criação dos Tensores Finais
    X_tr_t_seca, X_val_t_seca, X_tgt_t_seca = _to_tensor(X_tr_sc_seca, tempos_tr, num_features_seca), _to_tensor(X_val_sc_seca, tempos_val, num_features_seca), _to_tensor(X_tgt_sc_seca, tempos_tgt, num_features_seca)
    X_tr_t_pico, X_val_t_pico, X_tgt_t_pico = _to_tensor(X_tr_sc_pico, tempos_tr, num_features_pico), _to_tensor(X_val_sc_pico, tempos_val, num_features_pico), _to_tensor(X_tgt_sc_pico, tempos_tgt, num_features_pico)

    logpop_tr_t, logpop_val_t, logpop_tgt_t = _pop_tensor(df_tr_sort, tempos_tr), _pop_tensor(df_val_sort, tempos_val), _pop_tensor(df_tgt_sort, tempos_tgt)
    Y_tr_t, Y_val_t = _casos_tensor(df_tr_sort, tempos_tr), _casos_tensor(df_val_sort, tempos_val)
    Y_val_t_np = Y_val_t.cpu().numpy()

    # Limpeza de RAM agressiva
    del df_tr, df_tr_real, df_val_int, df_tr_sort, df_val_sort
    del X_tr_sc_seca, X_val_sc_seca, X_tgt_sc_seca, X_tr_sc_pico, X_val_sc_pico, X_tgt_sc_pico
    import gc
    gc.collect()

    # ==========================================================================
    # SUBFUNÇÃO DE TREINAMENTO DO ESPECIALISTA (Early Stopping via LOSS)
    # ==========================================================================
    def treinar_especialista(num_features, idx_v, idx_a, is_pico_mode, nome, X_tr_t, X_val_t):
        modelo = torch.compile(PreditorNBAGCRN(num_nodes, num_features, 64, idx_v, idx_a).to(device))

        # O modelo de Seca precisa de um weight_decay ligeiramente maior
        w_decay = 1e-4 if is_pico_mode else 1e-3
        otimizador = torch.optim.AdamW(modelo.parameters(), lr=0.004, weight_decay=w_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(otimizador, patience=5, factor=0.5, min_lr=1e-5)

        # 🔥 MUDANÇA: Monitorando a Loss ao invés do MAE
        melhor_loss_val = float('inf')
        sem_melhora = 0
        melhor_estado = None

        # [NOVO] checkpoint por época no Drive: se a sessão cair, rodar tudo de novo retoma daqui
        caminho_ckpt = f"ckpt_{nome}_s{season_id}.pt"
        epoca_ini = 0
        if RETOMAR and os.path.exists(caminho_ckpt):
            ck = torch.load(caminho_ckpt, map_location=device, weights_only=False)
            modelo.load_state_dict(ck['modelo'])
            otimizador.load_state_dict(ck['otimizador'])
            scheduler.load_state_dict(ck['scheduler'])
            melhor_loss_val, sem_melhora, melhor_estado = ck['melhor_loss_val'], ck['sem_melhora'], ck['melhor_estado']
            epoca_ini = ck['epoca'] + 1
            if ck['concluido']:
                print(f"   ♻️ {nome}: treino já concluído no checkpoint — usando os pesos salvos")
                if melhor_estado:
                    modelo.load_state_dict(melhor_estado)
                return modelo
            print(f"   ♻️ {nome}: retomando da época {epoca_ini} (melhor loss até aqui: {melhor_loss_val:.4f})")

        def _salvar_ckpt(epoca, concluido):
            torch.save({'modelo': modelo.state_dict(), 'otimizador': otimizador.state_dict(),
                        'scheduler': scheduler.state_dict(), 'melhor_loss_val': melhor_loss_val,
                        'sem_melhora': sem_melhora, 'melhor_estado': melhor_estado,
                        'epoca': epoca, 'concluido': concluido}, caminho_ckpt + '.tmp')
            os.replace(caminho_ckpt + '.tmp', caminho_ckpt)   # troca atômica: nunca fica checkpoint corrompido
            if concluido or (epoca + 1) % 5 == 0:
                salvar_no_drive(caminho_ckpt)                   # cópia no Drive a cada 5 épocas

        barra = tqdm(range(epoca_ini, epochs_max), desc=f' Treino {nome} (S{season_id})', unit='ep', file=sys.stdout,
                     initial=epoca_ini, total=epochs_max)
        epoch = epoca_ini - 1

        for epoch in barra:
            modelo.train()
            h = None

            for t in range(len(tempos_tr) - 1):
                otimizador.zero_grad()

                week_pura = epiweeks_tr[t + 1]
                pertence_ao_pico = (week_pura >= 47) or (week_pura <= 25)

                with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                    log_mu, log_r, h = modelo(X_tr_t[t], logpop_tr_t[t], h)
                    h = h.detach().clone()

                    if pertence_ao_pico == is_pico_mode:
                        alvo = Y_tr_t[t + 1].unsqueeze(-1)
                        if is_pico_mode:
                            loss = nb_loss_pico(log_mu, log_r, alvo)
                        else:
                            loss = nb_loss_seca(log_mu, log_r, alvo)

                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
                        otimizador.step()

            # --- Validação Estratificada (Calculando LOSS) ---
            modelo.eval()
            loss_val_total = 0.0
            mae_val_total = 0.0
            passos_validos = 0

            with torch.no_grad():
                h_val = None
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                    # Loop ajustado para len - 1, igual ao treino, para prever o tempo t+1
                    for t in range(len(tempos_val) - 1):
                        log_mu_v, log_r_v, h_val = modelo(X_val_t[t], logpop_val_t[t], h_val)
                        if h_val is not None: h_val = h_val.clone()

                        week_val = epiweeks_val[t + 1]
                        val_pertence_pico = (week_val >= 47) or (week_val <= 25)

                        if val_pertence_pico == is_pico_mode:
                            alvo_v = Y_val_t[t + 1].unsqueeze(-1)

                            # 1. Avalia a Loss (NLL + Penalidades)
                            if is_pico_mode:
                                loss_v = nb_loss_pico(log_mu_v, log_r_v, alvo_v)
                            else:
                                loss_v = nb_loss_seca(log_mu_v, log_r_v, alvo_v)

                            loss_val_total += loss_v.item()

                            # 2. Calcula MAE só para visualização no console
                            pred_mu = torch.exp(log_mu_v)
                            mae_v = torch.abs(pred_mu - alvo_v).mean().item()
                            mae_val_total += mae_v

                            passos_validos += 1

            if passos_validos > 0:
                loss_val_media = loss_val_total / passos_validos
                mae_val_media = mae_val_total / passos_validos

                # 🔥 MUDANÇA: Scheduler e Early Stopping agora dependem exclusivamente da LOSS
                scheduler.step(loss_val_media)
                barra.set_postfix({'Loss_V': f'{loss_val_media:.3f}', 'MAE_V': f'{mae_val_media:.1f}'})

                # Tolerância menor (1e-4) porque a Loss flutua em escalas decimais menores que o MAE absoluto
                if loss_val_media < melhor_loss_val - 1e-4:
                    melhor_loss_val = loss_val_media
                    melhor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
                    sem_melhora = 0
                else:
                    sem_melhora += 1
                    if sem_melhora >= patience: break

            if (epoch + 1) % CKPT_A_CADA_EPOCAS == 0:
                _salvar_ckpt(epoch, concluido=False)   # [NOVO]

        _salvar_ckpt(epoch, concluido=True)   # [NOVO]
        if melhor_estado:
            modelo.load_state_dict(melhor_estado)
            torch.save(melhor_estado, f"modelo_agcrn_{nome}_s{season_id}.pt")
        else:
            # [NOVO] sem nenhuma semana de validação (caso da SECA), salva os pesos da última época
            torch.save(modelo.state_dict(), f"modelo_agcrn_{nome}_s{season_id}.pt")

        return modelo

    # ==========================================================================
    # EXECUÇÃO DO TREINAMENTO E GERAÇÃO DO ALVO RECONCILIADO
    # ==========================================================================
    modelo_seca = treinar_especialista(num_features_seca, idx_vel_seca, idx_acel_seca, is_pico_mode=False, nome="SECA", X_tr_t=X_tr_t_seca, X_val_t=X_val_t_seca)
    modelo_pico = treinar_especialista(num_features_pico, idx_vel_pico, idx_acel_pico, is_pico_mode=True,  nome="PICO", X_tr_t=X_tr_t_pico, X_val_t=X_val_t_pico)

    modelo_seca.eval()
    modelo_pico.eval()
    preds_mu_tgt, preds_r_tgt = [], []

    with torch.no_grad():
        h_seca, h_pico = None, None
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for t in range(len(tempos_tgt)):
                log_mu_s, log_r_s, h_seca = modelo_seca(X_tgt_t_seca[t], logpop_tgt_t[t], h_seca)
                log_mu_p, log_r_p, h_pico = modelo_pico(X_tgt_t_pico[t], logpop_tgt_t[t], h_pico)

                if h_seca is not None: h_seca = h_seca.clone()
                if h_pico is not None: h_pico = h_pico.clone()

                week_pura = epiweeks_tgt[t]
                pertence_ao_pico = (week_pura >= 47) or (week_pura <= 25)

                if pertence_ao_pico:
                    preds_mu_tgt.append(torch.exp(log_mu_p).squeeze(-1))
                    preds_r_tgt.append(torch.exp(log_r_p).squeeze(-1))
                else:
                    preds_mu_tgt.append(torch.exp(log_mu_s).squeeze(-1))
                    preds_r_tgt.append(torch.exp(log_r_s).squeeze(-1))

    preds_mu_tgt_np = torch.stack(preds_mu_tgt).cpu().numpy()
    preds_r_tgt_np = torch.stack(preds_r_tgt).cpu().numpy()

    return {
        'season_id': season_id,
        'preds_mu_np': np.maximum(preds_mu_tgt_np, 0),
        'preds_r_np': np.maximum(preds_r_tgt_np, 1e-3),
        'df_tgt_sort': df_tgt_sort,
        'mapa_simples': mapa_simples,
    }

In [ ]:
# @title GERAÇÃO DA SUBMISSÃO MUNICIPAL (Vetorizada e Otimizada)
import pandas as pd
import numpy as np
from scipy.stats import nbinom

# ==============================================================================
# GERAÇÃO DA SUBMISSÃO MUNICIPAL (Vetorizada e Otimizada)
# ==============================================================================
def montar_submissao_agcrn(resultado_agcrn: dict, col_alvo: str = 'casos.x') -> pd.DataFrame:
    print("🛠️ Montando arquivo de submissão e calculando quantis...")

    df_tgt_sort  = resultado_agcrn['df_tgt_sort'].copy()

    # As matrizes mu e r vêm com formato [Semanas, Nós].
    # Como o df_tgt_sort foi perfeitamente ordenado por [time_index, node_id] no treino,
    # um simples .flatten() alinha os 300.000 tensores perfeitamente com as linhas do DataFrame.
    mu_flat = resultado_agcrn['preds_mu_np'].flatten()
    r_flat  = resultado_agcrn['preds_r_np'].flatten()

    # Prevenção matemática contra divisão por zero
    mu_flat = np.clip(mu_flat, a_min=1e-8, a_max=None)
    r_flat  = np.clip(r_flat, a_min=1e-8, a_max=None)

    # Probabilidade p requerida pela SciPy (nbinom)
    p_flat = r_flat / (r_flat + mu_flat)

    # Extrai o esqueleto base do DataFrame
    # Mapeie geocode para VARS['municipio'] caso seu pipeline posterior exija
    df_pred = df_tgt_sort[['geocode', 'date', 'epiweek', col_alvo]].copy()
    df_pred = df_pred.rename(columns={col_alvo: 'casos_reais'})

    # --------------------------------------------------------------------------
    # CÁLCULO VETORIZADO MASSIVO (Roda em milissegundos)
    # --------------------------------------------------------------------------
    quantis_necessarios = {
        'lower_95': 0.025, 'lower_90': 0.05, 'lower_80': 0.10, 'lower_50': 0.25,
        'pred': 0.50, # Mediana entra direto no loop
        'upper_50': 0.75,  'upper_80': 0.90, 'upper_90': 0.95, 'upper_95': 0.975,
    }

    for col_name, q in quantis_necessarios.items():
        # Calcula todos os 5492 nós x 52 semanas de uma única vez
        df_pred[col_name] = nbinom.ppf(q, n=r_flat, p=p_flat).astype(float)

    # Limitações físicas de predição (não existe caso negativo)
    cols_pred = ['pred'] + [k for k in quantis_necessarios.keys() if k != 'pred']
    df_pred[cols_pred] = df_pred[cols_pred].clip(lower=0).round(2)

    # --------------------------------------------------------------------------
    # AJUSTE DE DATAS E PREENCHIMENTO DE BURACOS (Grid Filler)
    # --------------------------------------------------------------------------
    # Parsing da data baseada no YYYYWW
    df_pred['date_parsed'] = pd.to_datetime(df_pred['epiweek'].astype(str) + '-0', format='%G%V-%w')
    df_pred['date'] = df_pred['date_parsed'].dt.strftime('%Y-%m-%d')

    # Garante que não falte nenhuma semana no calendário da submissão
    geocodes = df_pred['geocode'].unique()
    todas_datas = pd.date_range(start=df_pred['date_parsed'].min(), end=df_pred['date_parsed'].max(), freq='W-SUN')

    grade = pd.MultiIndex.from_product(
        [geocodes, todas_datas], names=['geocode', 'date_parsed']
    ).to_frame().reset_index(drop=True)
    grade['date'] = grade['date_parsed'].dt.strftime('%Y-%m-%d')

    # Realiza o Merge para criar buracos (NaN) onde faltarem semanas
    df_final = pd.merge(grade, df_pred, on=['geocode', 'date', 'date_parsed'], how='left')

    # Preenche os eventuais buracos com interpolação linear restrita
    for col in cols_pred:
        df_final[col] = df_final.groupby('geocode')[col].transform(lambda x: x.interpolate('linear').bfill().ffill())
        df_final[col] = np.maximum(df_final[col], 0).round(2)

    # Organização Final das Colunas
    colunas_finais = [
        'geocode', 'date', 'casos_reais', 'pred',
        'lower_50', 'upper_50', 'lower_80', 'upper_80',
        'lower_90', 'upper_90', 'lower_95', 'upper_95',
    ]

    # Usa geocode ou VARS['municipio'] conforme seu padrão
    return df_final[colunas_finais].sort_values(by=['geocode', 'date']).reset_index(drop=True)

## 6. Execução principal (célula 48)

In [ ]:
# @title RODAR O MODELO 2
if __name__ == '__main__':
    t_inicio = time.time()
    print(f"\n{'='*75}")
    print(f" 🚀 INICIANDO PIPELINE FINAL (DUPLO ESPECIALISTA): SEASON {SEASON_ALVO}")
    print(f"{'='*75}")

    db_conn = sqlite3.connect(DB_LOCAL)

    # [NOVO] caminho configurável (célula 2)
    arquivo_treino = os.path.join(PASTA_DADOS, f"train{SEASON_ALVO}_FINAL_NN.parquet")
    arquivo_teste  = os.path.join(PASTA_DADOS, f"test{SEASON_ALVO}_FINAL_NN.parquet")

    print(f"📥 Carregando tensores padronizados:\n   - {arquivo_treino}\n   - {arquivo_teste}")
    df_train = pd.read_parquet(arquivo_treino)
    df_test  = pd.read_parquet(arquivo_teste)

    df_train = df_train[df_train[VARS['uf_code']] != 32].reset_index(drop=True)
    df_test  = df_test[df_test[VARS['uf_code']] != 32].reset_index(drop=True)

    n_antes = len(df_train) + len(df_test)
    df_train = df_train.drop_duplicates(subset=[VARS['municipio'], VARS['data']]).reset_index(drop=True)
    df_test  = df_test.drop_duplicates(subset=[VARS['municipio'], VARS['data']]).reset_index(drop=True)
    print(f"🔪 Duplicatas removidas: {n_antes - len(df_train) - len(df_test)} (esperado 0 com os bancos corrigidos)")

    print("🧹 Compactando memória RAM (Downcasting)...")
    for df in [df_train, df_test]:
        colunas_float = df.select_dtypes(include=['float64']).columns
        df[colunas_float] = df[colunas_float].astype('float32')
        colunas_int = df.select_dtypes(include=['int64']).columns
        df[colunas_int] = df[colunas_int].astype('int32')

    mapa_simples = df_train[[VARS['municipio']]].drop_duplicates().sort_values(VARS['municipio']).reset_index(drop=True)
    mapa_simples['node_id'] = mapa_simples.index
    num_nodes = len(mapa_simples)
    print(f"🗺️ Malha Espacial validada: {num_nodes} municípios computáveis.")

    res_agcrn = treinar_e_validar_season(
        df_train=df_train,
        df_test=df_test,
        season_id=SEASON_ALVO,
        encoder_means_globais={},
        num_nodes=num_nodes,
        mapa_simples=mapa_simples,
        epochs_max=EPOCHS_MAX,
        patience=PATIENCE
    )
    print(f"⏱️ Treino concluído em {(time.time() - t_inicio) / 60:.1f} min")

    df_mun = montar_submissao_agcrn(res_agcrn, col_alvo=VARS['casos_dengue'])

    mapa_uf_code = df_train[[VARS['municipio'], VARS['uf_code']]].drop_duplicates()
    df_mun = df_mun.merge(mapa_uf_code, on=VARS['municipio'], how='left')

    print("📊 Agregando predições para a Macro-região Estadual (Baseline)...")
    cols_agregacao = ['pred', 'lower_50', 'upper_50', 'lower_80', 'upper_80',
                      'lower_90', 'upper_90', 'lower_95', 'upper_95', 'casos_reais']
    df_est = df_mun.groupby([VARS['uf_code'], 'date'])[cols_agregacao].sum().reset_index()

    df_mun_rec, df_est_rec = reconciliar_mint_dinamico(
        df_municipal=df_mun,
        df_estadual=df_est,
        semana_inicio_alta=47,
        semana_fim_alta=25,
        col_uf_code=VARS['uf_code']
    )

    print("💾 Salvando projeções reconciliadas no banco de dados SQLite...")
    df_mun_rec['season'] = SEASON_ALVO
    df_est_rec['season'] = SEASON_ALVO
    # [NOVO] apaga resultados anteriores DESTA season antes de gravar (antes era 'append' e misturava execuções)
    for tabela in ('municipal_results', 'estadual_results'):
        if db_conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (tabela,)).fetchone():
            db_conn.execute(f"DELETE FROM {tabela} WHERE season = ?", (SEASON_ALVO,))
    db_conn.commit()
    df_mun_rec.to_sql('municipal_results', db_conn, if_exists='append', index=False)
    df_est_rec.to_sql('estadual_results', db_conn, if_exists='append', index=False)
    db_conn.close()
    df_est_rec.to_csv(f"previsoes_estaduais_s{SEASON_ALVO}.csv", index=False)   # [NOVO] cópia legível
    for f in os.listdir('.'):                                                      # [NOVO] resultados -> Drive
        if not f.endswith('.parquet') and not f.endswith('.tmp'):
            salvar_no_drive(f)

    print(f"\n📦 Arquivos em {os.getcwd()}:")
    for f in sorted(os.listdir('.')):
        print(f"   - {f}  ({os.path.getsize(f) / 1e6:.1f} MB)")

    del df_train, df_test, res_agcrn, df_mun, df_est, df_mun_rec, df_est_rec
    gc.collect()
    print(f"\n✅ SEASON {SEASON_ALVO} COMPLETA em {(time.time() - t_inicio) / 60:.1f} min.")

## 7. Avaliação (célula 49) e gráfico por UF (célula 50)

In [ ]:
# @title Avaliar Desempenho
import sqlite3
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

def avaliar_desempenho_season(db_path: str = 'resultados_agcrn.db', season_id: int = 1):
    """
    Conecta ao banco de dados SQLite, extrai as predições reconciliadas (MinT)
    e calcula as métricas de validação científica (MAE, RMSE, sMAPE, Pearson)
    e Métricas Probabilísticas (Cobertura de Intervalos - PIC).
    """
    # 1. Conexão e extração de dados
    if not __import__('os').path.exists(db_path):
        print(f"❌ Banco de dados '{db_path}' não encontrado.")
        return

    conn = sqlite3.connect(db_path)
    df_mun = pd.read_sql(f"SELECT * FROM municipal_results WHERE season = {season_id}", conn)
    df_est = pd.read_sql(f"SELECT * FROM estadual_results WHERE season = {season_id}", conn)
    conn.close()

    if df_mun.empty or df_est.empty:
        print(f"❌ Nenhum registro encontrado para a Season {season_id} no banco '{db_path}'.")
        return

    # 2. Função matemática interna para o bloco de métricas (Incluindo Probabilidade)
    def _calcular_metricas(df):
        y_real = df['casos_reais'].fillna(0.0).values
        y_pred = df['pred'].fillna(0.0).values

        # Erros Absolutos e Quadráticos
        mae = mean_absolute_error(y_real, y_pred)
        rmse = np.sqrt(mean_squared_error(y_real, y_pred))

        # sMAPE protegido
        denominador = (np.abs(y_real) + np.abs(y_pred)) / 2
        smape = np.mean(np.where(denominador == 0, 0, np.abs(y_real - y_pred) / denominador)) * 100

        # Correlação de Pearson
        if np.std(y_real) == 0 or np.std(y_pred) == 0:
            pearson = 0.0
        else:
            pearson, _ = pearsonr(y_real, y_pred)

        # ======================================================================
        # CÁLCULO DE COBERTURA PROBABILÍSTICA (PIC)
        # Verifica se o valor real caiu dentro das bandas de confiança do modelo
        # ======================================================================
        coberturas = {}
        for conf in ['50', '80', '90', '95']:
            col_lower = f'lower_{conf}'
            col_upper = f'upper_{conf}'
            if col_lower in df.columns and col_upper in df.columns:
                dentro_do_intervalo = (df['casos_reais'] >= df[col_lower]) & (df['casos_reais'] <= df[col_upper])
                coberturas[conf] = dentro_do_intervalo.mean() * 100
            else:
                coberturas[conf] = 0.0

        return mae, rmse, smape, pearson, coberturas

    # 4. Execução dos cálculos
    mae_m, rmse_m, smape_m, pearson_m, cob_m = _calcular_metricas(df_mun)
    mae_e, rmse_e, smape_e, pearson_e, cob_e = _calcular_metricas(df_est)

    # 5. Exibição do Relatório com Rigor Acadêmico
    print(f"\n" + "="*70)
    print(f"      RELATÓRIO DE PERFORMANCE EPIDEMIOLÓGICA — SEASON {season_id}")
    print("="*70)

    print(f"📊 NÍVEL MUNICIPAL (Granularidade Espaço-Tempo Local):")
    print(f"  • MAE     : {mae_m:.2f} casos (Magnitude média do erro)")
    print(f"  • RMSE    : {rmse_m:.2f} casos (Sensível a erros em surtos máximos)")
    print(f"  • sMAPE   : {smape_m:.2f}% (Erro percentual relativo)")
    print(f"  • Pearson : {pearson_m:.4f} (Sincronia temporal da curva)")
    print(f"  --- Cobertura Probabilística (Incerteza do Modelo) ---")
    print(f"  • Casos dentro do IC 50% : {cob_m['50']:.1f}%")
    print(f"  • Casos dentro do IC 90% : {cob_m['90']:.1f}%")
    print(f"  • Casos dentro do IC 95% : {cob_m['95']:.1f}% (Meta teórica: ~95%)")

    print("\n" + "-"*70)

    print(f"🏛️ NÍVEL ESTADUAL (Agregação Macro Regional via MinT):")
    print(f"  • MAE     : {mae_e:.2f} casos")
    print(f"  • RMSE    : {rmse_e:.2f} casos")
    print(f"  • sMAPE   : {smape_e:.2f}%")
    print(f"  • Pearson : {pearson_e:.4f}")
    print(f"  --- Cobertura Probabilística Agregada ---")
    print(f"  • Casos dentro do IC 50% : {cob_e['50']:.1f}%")
    print(f"  • Casos dentro do IC 90% : {cob_e['90']:.1f}%")
    print(f"  • Casos dentro do IC 95% : {cob_e['95']:.1f}%")
    print("="*70 + "\n")

# ==============================================================================
# DISPARO DO SCRIPT DE ANÁLISE
# ==============================================================================
if __name__ == '__main__':
    # Altere o ID da Season conforme desejar avaliar
    avaliar_desempenho_season(db_path=DB_LOCAL, season_id=SEASON_ALVO)

In [ ]:
# @title GRÁFICO DE PREDIÇÕES ESTADUAIS
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def plotar_grid_geofacet_brasil_hd(df_estadual_resultados, season_id, col_uf=VARS['uf_code'], y_max_factor=1.5, y_max_absolute=None):
    """
    Gera o grid geofacet do Brasil com controle customizado do limite do Eixo Y
    para evitar que intervalos de confiança extremos prejudiquem a visualização.
    """
    df_plot = df_estadual_resultados.copy()

    # Garantindo que a data seja reconhecida corretamente
    df_plot['date'] = pd.to_datetime(df_plot['date'])
    df_plot = df_plot.sort_values('date')

    # Matriz do Layout Geográfico do Brasil
    grid_brasil = {
        14: {'uf': 'RR', 'row': 0, 'col': 1}, 16: {'uf': 'AP', 'row': 0, 'col': 4},
        12: {'uf': 'AC', 'row': 1, 'col': 0}, 13: {'uf': 'AM', 'row': 1, 'col': 1},
        11: {'uf': 'RO', 'row': 1, 'col': 2}, 15: {'uf': 'PA', 'row': 1, 'col': 3},
        17: {'uf': 'TO', 'row': 1, 'col': 4}, 21: {'uf': 'MA', 'row': 1, 'col': 5},
        23: {'uf': 'CE', 'row': 1, 'col': 6}, 24: {'uf': 'RN', 'row': 1, 'col': 7},
        51: {'uf': 'MT', 'row': 2, 'col': 2}, 52: {'uf': 'GO', 'row': 2, 'col': 3},
        53: {'uf': 'DF', 'row': 2, 'col': 4}, 22: {'uf': 'PI', 'row': 2, 'col': 5},
        26: {'uf': 'PE', 'row': 2, 'col': 6}, 25: {'uf': 'PB', 'row': 2, 'col': 7},
        50: {'uf': 'MS', 'row': 3, 'col': 2}, 31: {'uf': 'MG', 'row': 3, 'col': 3},
        29: {'uf': 'BA', 'row': 3, 'col': 4}, 27: {'uf': 'AL', 'row': 3, 'col': 5},
        28: {'uf': 'SE', 'row': 3, 'col': 6}, 35: {'uf': 'SP', 'row': 4, 'col': 3},
        33: {'uf': 'RJ', 'row': 4, 'col': 4}, 32: {'uf': 'ES', 'row': 4, 'col': 5},
        41: {'uf': 'PR', 'row': 5, 'col': 2}, 42: {'uf': 'SC', 'row': 5, 'col': 3},
        43: {'uf': 'RS', 'row': 6, 'col': 2}
    }

    fig, axes = plt.subplots(7, 8, figsize=(24, 20))
    for ax in axes.flat: ax.axis('off')

    for code, info in grid_brasil.items():
        ax = axes[info['row'], info['col']]
        df_uf = df_plot[df_plot[col_uf] == code].dropna(subset=['pred', 'date'])

        if df_uf.empty: continue

        ax.axis('on')
        # Plot das bandas de incerteza
        ax.fill_between(df_uf['date'], df_uf['lower_95'], df_uf['upper_95'],
                        color='#1e88e5', alpha=0.15, label='IC 95%' if code==14 else "")

        # Plot da predição reconciliada
        ax.plot(df_uf['date'], df_uf['pred'], color='#00468b', linewidth=1.5,
                label='Predito (MinT)' if code==14 else "")

        # Plot dos dados reais
        ax.plot(df_uf['date'], df_uf['casos_reais'], color='#d32f2f', linewidth=1.2, linestyle=':',
                label='Real' if code==14 else "")

        ax.set_title(f"📍 {info['uf']}", fontsize=10, weight='bold')
        ax.tick_params(axis='both', which='major', labelsize=7)

        # ----------------------------------------------------------------------
        # CONTROLE DO EIXO Y (Ignora os picos do IC95% na escala do gráfico)
        # ----------------------------------------------------------------------
        if y_max_absolute is not None:
            ax.set_ylim(bottom=0, top=y_max_absolute)
        elif y_max_factor is not None:
            # Encontra o pico real da epidemia no estado (linha central ou casos reais)
            pico_visivel = max(df_uf['pred'].max(), df_uf['casos_reais'].max())
            ax.set_ylim(bottom=0, top=pico_visivel * y_max_factor)
        else:
            ax.set_ylim(bottom=0)

    fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.95), fontsize=12)
    plt.suptitle(f'🗺️ Validação Hierárquica: AGCRN + Dynamic MinT (Season {season_id})', fontsize=20, weight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    nome_arquivo = f'validacao_brasil_s{season_id}_4K.png'
    plt.savefig(nome_arquivo, dpi=200, bbox_inches='tight')
    plt.show()

    salvar_no_drive(nome_arquivo)   # [NOVO] cópia no Drive, sem download automático
    print(f"🖼️ Gráfico salvo em {os.path.join(PASTA_SAIDA, nome_arquivo)}")

# Execução
import sqlite3
db_conn = sqlite3.connect(DB_LOCAL)

# Carrega a última submissão para a temporada atual
df_est_rec = pd.read_sql_query(f"SELECT * FROM estadual_results WHERE season = {SEASON_ALVO}", db_conn)
db_conn.close()

# Agora sim, chame a função de plotagem
plotar_grid_geofacet_brasil_hd(df_est_rec, season_id=SEASON_ALVO)

## 8. (Opcional) Referência: persistência ingênua

"A semana que vem terá os mesmos casos desta semana", no nível estadual, com os mesmos dados de teste.
Os modelos recebem os casos observados de cada semana do teste, então este é o mínimo que eles precisam superar.
Na avaliação feita no mapa do projeto, deu MAE ≈ 197 e r ≈ 0,98 nos bancos antigos.

In [ ]:
# @title Persistência ingênua (nível estadual)
tr_ = pd.read_parquet(os.path.join(PASTA_DADOS, f"train{SEASON_ALVO}_FINAL_NN.parquet"), columns=['uf_code', 'date', 'casos.x'])
te_ = pd.read_parquet(os.path.join(PASTA_DADOS, f"test{SEASON_ALVO}_FINAL_NN.parquet"), columns=['uf_code', 'date', 'casos.x'])
est_te = te_.groupby(['uf_code', 'date'])['casos.x'].sum().reset_index().sort_values(['uf_code', 'date'])
ultimo_tr = tr_.groupby(['uf_code', 'date'])['casos.x'].sum().reset_index().sort_values('date').groupby('uf_code')['casos.x'].last()
est_te['persist'] = est_te.groupby('uf_code')['casos.x'].shift(1)
est_te['persist'] = est_te['persist'].fillna(est_te['uf_code'].map(ultimo_tr))
y, p = est_te['casos.x'].to_numpy(float), est_te['persist'].to_numpy(float)
den = (np.abs(y) + np.abs(p)) / 2
print(f"Persistência estadual S{SEASON_ALVO}: MAE {np.mean(np.abs(y - p)):.2f} · RMSE {np.sqrt(np.mean((y - p) ** 2)):.2f} · "
      f"sMAPE {np.mean(np.where(den == 0, 0, np.abs(y - p) / np.where(den == 0, 1, den))) * 100:.2f}% · r {pearsonr(y, p)[0]:.4f}")
del tr_, te_